# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a worked example for loading and exploring the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their '@id'
print("Available record sets (@id and name):")
record_sets = []
for record_set in dataset.record_sets():
    print(f"  @id: {record_set['@id']}  |  name: {record_set.get('name', '')}")
    record_sets.append(record_set)

# If record sets are available, list all fields and their '@id's for the first one
if record_sets:
    main_record_set = record_sets[0]
    print(f"\nFields for record set: {main_record_set['@id']} ({main_record_set.get('name','')})")
    for field in main_record_set.get('field', []):
        print(f"  @id: {field['@id']} | name: {field.get('name', '')}")
else:
    print("No record sets found in this dataset. Please check the schema or dataset definition.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data for each identified record set
record_set_ids = [r['@id'] for r in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}.")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Print the fields for the first available DataFrame
if dataframes:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping.

In [ ]:
import numpy as np

# For demonstration, let's pick "Age" as a numeric field if present (field @id should be used)
main_numeric_field_id = None
group_field_id = None

# Try to infer the field IDs for 'Age' and a grouping field such as 'Sex' using the first record set
if record_sets:
    fields = record_sets[0].get('field', [])
    for f in fields:
        # Attempt to match on field names (fallback in absence of documentation)
        n = f.get('name','').lower()
        if n == 'age' or 'age' in n:
            main_numeric_field_id = f['@id']
        if group_field_id is None and (n == 'sex' or 'sex' in n or 'gender' in n):
            group_field_id = f['@id']

    print(f"Numeric field (age) @id: {main_numeric_field_id}")
    print(f"Group field ('sex') @id: {group_field_id}")

if not main_numeric_field_id or not group_field_id:
    print("Could not automatically find suitable field IDs for numeric or group field. Please specify manually if needed.")

# Proceed only if a suitable numeric field was found
if main_rs_id in dataframes and main_numeric_field_id in dataframes[main_rs_id].columns:
    threshold = 50    # Example age threshold
    df = dataframes[main_rs_id]

    # Convert to numeric, handle errors
    df[main_numeric_field_id] = pd.to_numeric(df[main_numeric_field_id], errors='coerce')
    filtered_df = df[df[main_numeric_field_id] > threshold]
    print(f"Filtered records with {main_numeric_field_id} > {threshold} (total: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{main_numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[main_numeric_field_id] - filtered_df[main_numeric_field_id].mean()) / filtered_df[main_numeric_field_id].std()
    print(f"\nNormalized {main_numeric_field_id} for filtered records:")
    display(filtered_df[[main_numeric_field_id, norm_col]].head())

    # Group by group_field_id (e.g., "sex") if present, and show mean age by group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[main_numeric_field_id].mean().to_frame('mean_' + main_numeric_field_id)
        print(f"\nMean {main_numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field (e.g. 'Age') found for EDA or data could not be loaded. Skipping EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if we had a valid numeric field and filtered_df
if 'filtered_df' in locals() and not filtered_df.empty and main_numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[main_numeric_field_id], bins=12, kde=True)
    plt.title(f"Distribution of {main_numeric_field_id} (Filtered >{threshold})")
    plt.xlabel(main_numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, plot boxplot
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[main_numeric_field_id])
        plt.title(f"{main_numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(main_numeric_field_id)
        plt.show()
else:
    print("No data available for visualization.")


## 6. Conclusion
This notebook demonstrated end-to-end exploration of the FAIR^2 colorectal cancer survivor dataset using the `mlcroissant` library. We loaded metadata via Croissant schema, inspected record set and field IDs (referencing everything by `@id`), loaded records into pandas, performed EDA with normalization and grouping, and visualized numeric variables.

**Key observations:**
- Dataset fields should always be referenced using their Croissant `@id` identifier for consistency and schema-independence.
- The `mlcroissant` library enables robust loading and scalable inspection of FAIR datasets described with the Croissant standard.
- Analytical and visualization best practices prepare the dataset for downstream clinical or ML analysis.

Further analysis can include more detailed biostatistical summaries or machine learning applications using the Croissant-enabled workflow.